# BRACS SL Test: Slide-Level Evaluation

Slide-level evaluation of the breast carcinoma models on **BRACS** (external dataset), Full Rescaled & Filtered &
Non-Atypical, **SL test split**: 323 slides (163 carcinoma: 113 IC + 50 DCIS; 160 non-carcinoma: 123 PB + 37 N).

Inputs are the per-slide prediction tables of the SL test runs (experiment 170 on the external MLflow):

* **TL models** (tile classifiers + heuristic aggregation): one run per model, four aggregators (Max, MPM, Top-k,
  Majority Vote of the three).
* **MIL models** (Classic / Hybrid MIL on PGP / Virchow2 embeddings): one slide score per slide.

All decisions use the **in-house-tuned thresholds** stored in each run's config; nothing is re-tuned on BRACS.

The notebook

1. plots the slide score histograms per BRACS subtype, with the in-house threshold,
2. recomputes the slide-level metrics **without the DCIS slides** (sanity-checked against the logged metrics on all
   slides) and writes them as a "Without DCIS slides" section of the result pages.

**Running:** kernel `Python (carcinoma)`. Set `EXTERNAL_MLFLOW_TRACKING_USERNAME` and
`EXTERNAL_MLFLOW_TRACKING_PASSWORD` (external MLflow credentials) before starting the kernel.

## Setup

In [ ]:
import json
import os
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import yaml
from IPython.display import Markdown, display
from sklearn.metrics import average_precision_score, roc_auc_score


# BRACS runs live on the external MLflow server; credentials come from the environment.
EXTERNAL_MLFLOW_URI = "https://mlflow.rationai.cloud.e-infra.cz"
os.environ["MLFLOW_TRACKING_USERNAME"] = os.environ["EXTERNAL_MLFLOW_TRACKING_USERNAME"]
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.environ["EXTERNAL_MLFLOW_TRACKING_PASSWORD"]
mlflow.set_tracking_uri(EXTERNAL_MLFLOW_URI)

# Folder with the result pages (tl_page_bracs.md, mil_page_bracs.md) to update in place.
# None = only display the generated sections.
PAGES_DIR: Path | None = None

### Runs and data

Row names match the result pages. TL runs are listed in page order; MIL runs carry their model registry version
(used for the row links).

In [ ]:
# SL test runs of the TL models: page row name -> run id
TL_RUNS = {
    "VGG16": "bc00f145852d46e39ac78bf9b11c5d4b",
    "VGG16-stain": "b48fcab84df540aeb14ec0ac18f9bba2",
    "ResNet50": "5ba0503ce435491185b02d0827a65f7b",
    "ResNet50-stain": "5f815854c6194eee95a329044df138a8",
    "PGP (frozen)": "88ce26d868ba4e118450ca809920c606",
    "PGP-stain (frozen)": "0811343c1c834bf8b806224d2d449b5c",
    "Virchow2 (frozen)": "999490b711c64c85a39a946864b8dbec",
    "Virchow2-stain (frozen)": "fa41ff1057114c9e8acc6f23c01f8e2d",
    "GViT Stain": "2d3c172669e144599b236df953c4ddc9",
    "GViT": "5915e9fe1dea42c6a721d0deb97216b1",
}
# Aggregators of the TL runs, in the order of the page cells (Max / MPM / Top-k / Majority Vote)
TL_AGGREGATORS = ["max", "mean_pool_max", "top_k", "majority_vote"]

# SL test runs of the MIL models: page row name -> (run id, model registry version)
MIL_RUNS = {
    "Hybrid MIL PGP": ("242fe0eb0be6484dae0ac87b341bc14f", 12),
    "Hybrid MIL Virchow2": ("99472290df374c1498949c4b95a4f21b", 11),
    "Classic MIL PGP": ("4471e2c1ca224ae584a29b883767f1ce", 14),
    "Classic MIL Virchow2": ("9ec8a90ead714a05aaf28f23890facd4", 13),
}
REGISTRY_URL = (
    f"{EXTERNAL_MLFLOW_URI}/#/models/Breast%20cancer%20binary%20classifier/versions"
)

# Slide metadata of the SL test split
# (configs/data/breast/bracs/full_rescaled_filtered_sl_test.yaml -> metadata_table)
METADATA_URI = "mlflow-artifacts:/170/7d532459cca5489eaa4dff365e4e1ef0/artifacts/bracs_rescaled_filtered_non_atypical_test.csv"
SUBTYPE_ORDER = ["IC", "DCIS", "PB", "N"]  # IC, DCIS = carcinoma; PB, N = non-carcinoma
SUBTYPE_COLORS = ["#b2182b", "#ef8a62", "#67a9cf", "#2166ac"]

## Loading predictions and in-house thresholds

In [ ]:
def download(run_id: str, path: str, read):
    """Download one artifact of a run to a temporary folder and read it with `read(file)`."""
    with (
        tempfile.TemporaryDirectory() as d,
        open(
            mlflow.artifacts.download_artifacts(
                run_id=run_id, artifact_path=path, dst_path=d
            )
        ) as f,
    ):
        return read(f)


with tempfile.TemporaryDirectory() as d:
    metadata = pd.read_csv(
        mlflow.artifacts.download_artifacts(METADATA_URI, dst_path=d)
    )
SUBTYPES = metadata.set_index("WSI Filename")["WSI label"]
print(SUBTYPES.value_counts().to_dict())


def load_table(run_id: str, table: str) -> pd.DataFrame:
    """Per-slide predictions with unified columns: slide, score, binary (if the table has a score), target, subtype."""
    j = download(run_id, f"tables/{table}", json.load)
    df = pd.DataFrame(j["data"], columns=j["columns"])
    # TL tables use slide_name/prediction(_binary), MIL tables slide/sl_prediction(_binary).
    # The majority vote table has no score: its "prediction" is already the binary decision.
    df = df.rename(
        columns={
            "slide_name": "slide",
            "prediction": "score",
            "prediction_binary": "binary",
            "sl_prediction": "score",
            "sl_prediction_binary": "binary",
        }
    )
    df["subtype"] = df["slide"].map(SUBTYPES)
    assert df["subtype"].notna().all(), (
        f"{run_id}/{table}: slides without a BRACS subtype"
    )
    return df


def tl_table(name: str, aggregator: str) -> pd.DataFrame:
    return load_table(TL_RUNS[name], f"sl_predictions_{aggregator}.json")


def mil_table(name: str) -> pd.DataFrame:
    return load_table(MIL_RUNS[name][0], "sl_predictions.json")


# Where each run's resolved config stores the in-house-tuned decision threshold
CONFIG_THRESHOLD_KEYS = {
    "max": ("model_data", "slide_max_threshold"),
    "mean_pool_max": ("model_data", "slide_mpm_threshold"),
    "top_k": ("model_data", "slide_top_k_threshold"),
    "mil": ("trainer", "callbacks", "mil_prediction_callback", "threshold"),
}


def in_house_threshold(run_id: str, aggregator: str) -> float:
    """Decision threshold tuned on in-house data, read from the run's resolved config."""
    value = download(run_id, "configs/config-resolved.yaml", yaml.safe_load)
    for key in CONFIG_THRESHOLD_KEYS[aggregator]:
        value = value[key]
    return float(value)


def check_decisions(df: pd.DataFrame, threshold: float) -> None:
    """The logged binary decisions must be exactly score > in-house threshold."""
    mismatches = ((df["score"] > threshold) != df["binary"].astype(bool)).sum()
    assert mismatches == 0, (
        f"{mismatches} logged decisions disagree with the in-house threshold {threshold}"
    )

## Score histograms per BRACS subtype

Stacked histograms of the slide scores, coloured by BRACS subtype; the dashed line is the in-house threshold.

In [ ]:
# "Nines" scale for TL scores, which saturate close to 1: score -> -log10(1 - score), so 0.9 -> 1, 0.99 -> 2, ...
NINES_EPS = 1e-8


def to_nines(x):
    return -np.log10(np.clip(1 - np.asarray(x, dtype=float), NINES_EPS, 1))


def plot_hist(
    ax,
    df: pd.DataFrame,
    threshold: float,
    title: str,
    bins: int = 40,
    scale: str = "linear",
) -> None:
    """Stacked histogram of slide scores per subtype; scale is "linear" ([0, 1]) or "nines"."""
    if scale == "nines":
        x = to_nines
        edges = np.linspace(0, to_nines(df["score"]).max() + 0.25, bins + 1)
        xlabel = "slide-level score (nines scale)"
    else:
        x = np.asarray
        edges = np.linspace(0, 1, bins + 1)
        xlabel = "slide-level score"

    ax.hist(
        [x(df.loc[df["subtype"] == s, "score"]) for s in SUBTYPE_ORDER],
        bins=edges,
        stacked=True,
        color=SUBTYPE_COLORS,
        label=SUBTYPE_ORDER,
    )
    ax.axvline(
        x(threshold),
        color="black",
        linestyle="--",
        linewidth=1,
        label=f"in-house threshold {threshold:.6g}",
    )

    if scale == "nines":
        ticks = np.arange(0, int(edges[-1]) + 1)
        labels = ["0"] + [f"0.{'9' * k}" if k <= 4 else f"1-1e-{k}" for k in ticks[1:]]
        ax.set_xticks(ticks, labels, fontsize=7)
    ax.set_xlim(edges[0], edges[-1])
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("# slides")
    ax.legend(fontsize=7)

In [ ]:
# MIL models: one histogram per model
fig, axes = plt.subplots(1, len(MIL_RUNS), figsize=(5 * len(MIL_RUNS), 4), sharey=True)
for ax, (name, (run_id, _)) in zip(axes, MIL_RUNS.items(), strict=True):
    df, threshold = mil_table(name), in_house_threshold(run_id, "mil")
    check_decisions(df, threshold)
    plot_hist(ax, df, threshold, name)
fig.suptitle("BRACS SL score histograms: MIL models")
fig.tight_layout()
plt.show()

In [ ]:
# TL models: one row per model, one column per aggregator, on the nines scale
# (majority vote has no continuous score and is skipped)
SCORE_AGGREGATORS = [a for a in TL_AGGREGATORS if a != "majority_vote"]

fig, axes = plt.subplots(
    len(TL_RUNS),
    len(SCORE_AGGREGATORS),
    figsize=(5 * len(SCORE_AGGREGATORS), 3.5 * len(TL_RUNS)),
)
for row, (name, run_id) in zip(axes, TL_RUNS.items(), strict=True):
    for ax, aggregator in zip(row, SCORE_AGGREGATORS, strict=True):
        df, threshold = (
            tl_table(name, aggregator),
            in_house_threshold(run_id, aggregator),
        )
        check_decisions(df, threshold)
        plot_hist(ax, df, threshold, f"{name} / {aggregator}", scale="nines")
fig.suptitle("BRACS SL score histograms: TL models", y=1.0)
fig.tight_layout()
plt.show()

## Metrics without DCIS slides

The DCIS slides (all carcinoma) are dropped and the slide-level metrics are recomputed from the per-slide predictions
at the in-house thresholds. Majority vote combines the three TL decisions and has no threshold of its own, so its
logged decision is used (and it has no AUROC/AUPRC).

Sanity check: on all slides, the recomputed metrics must reproduce the metrics logged by each run
(`sl/<aggregator>/<metric>` for TL, `sl/mil/<metric>` for MIL).

In [ ]:
METRICS = [
    "AUC",
    "AUPRC",
    "accuracy",
    "precision",
    "recall",
    "specificity",
    "negative_predictive_value",
]


def sl_metrics(df: pd.DataFrame, threshold: float | None) -> dict:
    """Slide-level metrics; carcinoma if score > threshold. threshold=None: majority vote (logged decision)."""
    y = df["target"].astype(bool)
    pred = df["score"].astype(bool) if threshold is None else df["score"] > threshold
    tp, fp = int((pred & y).sum()), int((pred & ~y).sum())
    tn, fn = int((~pred & ~y).sum()), int((~pred & y).sum())

    def ratio(a: int, b: int) -> float:
        return a / b if b else 0.0

    return {
        "AUC": np.nan if threshold is None else roc_auc_score(y, df["score"]),
        "AUPRC": np.nan
        if threshold is None
        else average_precision_score(y, df["score"]),
        "accuracy": ratio(tp + tn, len(df)),
        "precision": ratio(tp, tp + fp),
        "recall": ratio(tp, tp + fn),
        "specificity": ratio(tn, tn + fp),
        "negative_predictive_value": ratio(tn, tn + fn),
    }


def check_logged(
    df: pd.DataFrame, threshold: float | None, run_id: str, prefix: str
) -> None:
    """On all slides, the recomputed metrics must match the ones logged by the run under `prefix`."""
    logged = mlflow.get_run(run_id).data.metrics
    full = sl_metrics(df, threshold)
    diffs = {
        m: abs(full[m] - logged[f"{prefix}/{m}"])
        for m in METRICS
        if f"{prefix}/{m}" in logged
    }
    assert diffs, f"{run_id}: no logged metrics under {prefix}"
    assert max(diffs.values()) < 1e-3, (
        f"{run_id} {prefix}: recomputed metrics differ from the logged ones {diffs}"
    )


def evaluate_without_dcis(
    df: pd.DataFrame, threshold: float | None, run_id: str, prefix: str
) -> dict:
    if threshold is not None:
        check_decisions(df, threshold)
    check_logged(df, threshold, run_id, prefix)
    kept = df[df["subtype"] != "DCIS"]
    slide_counts.add(
        (
            len(df),
            len(kept),
            int(kept["target"].sum()),
            int((~kept["target"].astype(bool)).sum()),
        )
    )
    return sl_metrics(kept, threshold)


slide_counts = set()  # (all, kept, kept positive, kept negative) per table; must be the same for all models

tl_no_dcis = {}
for name, run_id in TL_RUNS.items():
    for aggregator in TL_AGGREGATORS:
        threshold = (
            None
            if aggregator == "majority_vote"
            else in_house_threshold(run_id, aggregator)
        )
        tl_no_dcis[(name, aggregator)] = evaluate_without_dcis(
            tl_table(name, aggregator), threshold, run_id, f"sl/{aggregator}"
        )

mil_no_dcis = {}
for name, (run_id, _) in MIL_RUNS.items():
    mil_no_dcis[name] = evaluate_without_dcis(
        mil_table(name), in_house_threshold(run_id, "mil"), run_id, "sl/mil"
    )

assert len(slide_counts) == 1, (
    f"models were evaluated on different slides: {slide_counts}"
)
N_ALL, N_KEPT, N_POS, N_NEG = slide_counts.pop()
print(
    f"slides: {N_ALL} -> {N_KEPT} without DCIS ({N_POS} carcinoma, {N_NEG} non-carcinoma)"
)

tl_no_dcis = pd.DataFrame(tl_no_dcis).T[METRICS]
mil_no_dcis = pd.DataFrame(mil_no_dcis).T[METRICS]
display(tl_no_dcis.style.format(precision=3))
display(mil_no_dcis.style.format(precision=3))

### Result pages

Builds the "Without DCIS slides" sections and inserts them before the summary / conclusion heading of each page
(an existing section is replaced). With `PAGES_DIR = None` the sections are only displayed.

In [ ]:
SECTION = "## Without DCIS slides"
HEADER = "| Model | AUROC | AUPRC | Accuracy | Precision | Recall | Specificity | Negative Predictive Score |"
INTRO = (
    f"Same models, per-slide predictions and decision thresholds as above, but the DCIS slides are excluded "
    f"({N_ALL - N_KEPT} slides, all carcinoma). Remaining: {N_KEPT} slides "
    f"({N_POS} carcinoma = IC, {N_NEG} non-carcinoma = PB + N)."
)


def fmt(value: float) -> str:
    return "\\-" if np.isnan(value) else f"{value:.3f}"


def write_section(page: str, section: list[str], before: str) -> None:
    """Insert `section` before the `before` heading of the page (replacing an existing one), or just display it."""
    display(Markdown("\n".join(section)))
    if PAGES_DIR is None:
        return
    path = Path(PAGES_DIR) / page
    text = path.read_text()
    if SECTION in text:  # drop the old section, up to the next "## " heading
        start = text.index(SECTION)
        end = text.find("\n## ", start + len(SECTION))
        text = text[:start] + (text[end + 1 :] if end != -1 else "")
    assert before in text, f"{page}: heading {before!r} not found"
    path.write_text(text.replace(before, "\n".join(section) + "\n\n" + before, 1))
    print(f"updated {page}")


# TL page: one cell per metric with the four aggregators stacked (Max / MPM / Top-k / Majority Vote)
tl_rows = [
    f"| {name} | "
    + " | ".join(
        "<br />".join(fmt(tl_no_dcis.loc[(name, a), m]) for a in TL_AGGREGATORS)
        for m in METRICS
    )
    + " |"
    for name in TL_RUNS
]
write_section(
    "tl_page_bracs.md",
    [
        SECTION,
        "",
        INTRO,
        "",
        "### **Metrics (Max / MPM / Top-k / Majority Vote)**",
        "",
        HEADER,
        "| --- | --- | --- | --- | --- | --- | --- | --- |",
        *tl_rows,
    ],
    before="## Summary",
)

# MIL page: model registry link + run link per row
mil_rows = [
    f"| [{name}]({REGISTRY_URL}/{version}) | "
    + " | ".join(fmt(mil_no_dcis.loc[name, m]) for m in METRICS)
    + f" | [URL]({EXTERNAL_MLFLOW_URI}/#/experiments/170/runs/{run_id}) |"
    for name, (run_id, version) in MIL_RUNS.items()
]
write_section(
    "mil_page_bracs.md",
    [
        SECTION,
        "",
        INTRO,
        "",
        HEADER + " MLFlow Run |",
        "| --- | --- | --- | --- | --- | --- | --- | --- | --- |",
        *mil_rows,
    ],
    before="## Conclusion(s)",
)